## **Flipkart Mobile Price Intelligence & Market Analysis**

### **Data cleaning & preparation**

#### **1. Importing the Required Libraries**

In [68]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

#### **2. Loading the dataset**

In [69]:
df = pd.read_csv('/content/flipkart_mobiles_raw.csv')

#### **3. Viewing First 10 rows**

In [70]:
df.head(10)

,Brand,Model,Color,Memory,Storage,Rating,Selling Price,Original Price
0,OPPO,A53,Moonlight Black,4 GB,64 GB,4.5,11990,15990
1,OPPO,A53,Mint Cream,4 GB,64 GB,4.5,11990,15990
2,OPPO,A53,Moonlight Black,6 GB,128 GB,4.3,13990,17990
3,OPPO,A53,Mint Cream,6 GB,128 GB,4.3,13990,17990
4,OPPO,A53,Electric Black,4 GB,64 GB,4.5,11990,15990
5,OPPO,A53,Electric Black,6 GB,128 GB,4.3,13990,17990
6,OPPO,A12,Deep Blue,4 GB,64 GB,4.4,10490,11990
7,OPPO,A12,Black,3 GB,32 GB,4.4,9490,10990
8,OPPO,A12,Blue,3 GB,32 GB,4.4,9490,10990
9,OPPO,A12,Flowing Silver,3 GB,32 GB,4.4,9490,10990


#### **4. Understanding the columns & Cleaning them**

In [71]:
df.columns

Index(['Brand', 'Model', 'Color', 'Memory', 'Storage', 'Rating',
       'Selling Price', 'Original Price'],
      dtype='object')

In [72]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3114 entries, 0 to 3113
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Brand           3114 non-null   object 
 1   Model           3114 non-null   object 
 2   Color           3114 non-null   object 
 3   Memory          3071 non-null   object 
 4   Storage         3075 non-null   object 
 5   Rating          2970 non-null   float64
 6   Selling Price   3114 non-null   int64  
 7   Original Price  3114 non-null   int64  
dtypes: float64(1), int64(2), object(5)
memory usage: 194.8+ KB


##### **4.1 Brand**

In [73]:
df['Brand'].nunique()

17

In [74]:
df['Brand'].unique()

array(['OPPO', 'HTC', 'IQOO', 'Google Pixel', 'LG', 'ASUS', 'realme',
       'GIONEE', 'Nokia', 'Apple', 'SAMSUNG', 'Lenovo', 'Motorola',
       'POCO', 'vivo', 'Xiaomi', 'Infinix'], dtype=object)

Insights:
* No null value in this column
* No 'Unknown' value in this column
* There are totally `17` brands of mobiles

##### **4.2 Model**

In [75]:
df['Model'].nunique()

914

In [76]:
df[df['Model']=='Unknown']

,Brand,Model,Color,Memory,Storage,Rating,Selling Price,Original Price


Insights:
* No 'Unknown' value in this column
* There are totally `914` models of mobiles

##### **4.3 Color**

In [77]:
df['Color'].nunique()

639

In [78]:
df[df['Color']=='Unknown']

,Brand,Model,Color,Memory,Storage,Rating,Selling Price,Original Price


Insights:
* No 'Unknown' value in this column
* There are totally `639` colors of mobiles

##### **4.4 Memory**

In [79]:
df['Memory'].unique()


array(['4 GB', '6 GB', '3 GB', '8 GB', '2 GB', '12 GB', '1 GB', '512 MB',
       '1.5 GB', '768 MB', '16 GB', '18 GB', '8 MB', '64 MB', '4 MB',
       '32 MB', '16 MB', '128 MB', nan, '4GB', '153 MB', '2 MB', '10 MB',
       '46 MB', '32 GB', '100 MB', '30 MB'], dtype=object)

In [80]:
df['Memory'].isnull().sum()

np.int64(43)

Insights:
* There '43' null values --> convert it to 'Unknown' --> then to '0'
* 'Unknown' value is present in this column --> needs to replaced with '0'
* Memory values are in different capacities such as 'GB', 'MB' --> needs to convert into single format as 'GB' only
* There are different formats such as '4GB' without gap & '4 GB' with gap --> need to attention while separating the digits.
* Convert the data type to Integer.


##### **4.5 Storage**

In [81]:
df['Storage'].unique()

array(['64 GB', '128 GB', '32 GB', '256 GB', '16 GB', '8 GB', '4 GB',
       '512 GB', nan, '16 MB', '128 MB', '2 MB', '4 MB', '48 MB', '8 MB',
       'Expandable Upto 32 GB', 'Expandable Upto 16 GB', '10 MB',
       '256 MB', '140 MB', '64 MB', '1 TB', '153 MB', '512 MB', '100 MB',
       '129 GB', '130 GB'], dtype=object)

In [82]:
df['Storage'].isnull().sum()

np.int64(39)

Insights:

* There '39' null values --> convert it to 'Unknown' --> then to '0'
* 'Unknown' value is present in this column --> needs to replaced with '0'
* Memory values are in different capacities such as 'GB', 'MB', 'TB' --> needs to convert into single format as 'GB' only
* There are different formats such as '4GB' without gap & '4 GB' with gap & also 'Expandable Upto 32 GB' --> need to attention while separating the digits.
* Convert the data type to Integer.

In [83]:
# Fill missing values
df['Memory'] = df['Memory'].fillna("Unknown")
df['Storage'] = df['Storage'].fillna("Unknown")

In [84]:
import re

def clean_memory_storage(value):
    if not isinstance(value, str):
        return 0

    # Remove spaces and convert to lowercase
    v = value.lower().replace(" ", "")

    # Case: Unknown
    if "unknown" in v:
        return 0

    # Case: expandable values → extract only number
    # e.g. "expandableupto32gb" → "32"
    if "expandable" in v:
        nums = re.findall(r'\d+', v)
        return int(nums[0]) if nums else 0

    # Case: GB (normal or messy)
    if "gb" in v:
        nums = re.findall(r'\d+\.?\d*', v)
        return float(nums[0]) if nums else 0

    # Case: MB → convert to GB
    if "mb" in v:
        nums = re.findall(r'\d+\.?\d*', v)
        if nums:
            return float(nums[0]) / 1024  # convert MB to GB
        return 0

    # Case: TB → convert to GB
    if "tb" in v:
        nums = re.findall(r'\d+\.?\d*', v)
        if nums:
            return float(nums[0]) * 1024  # TB to GB
        return 0

    return 0

# Apply function
df['Memory_GB'] = df['Memory'].apply(clean_memory_storage)
df['Storage_GB'] = df['Storage'].apply(clean_memory_storage)

df[['Memory', 'Memory_GB', 'Storage', 'Storage_GB']].head(5)


,Memory,Memory_GB,Storage,Storage_GB
0,4 GB,4.0,64 GB,64.0
1,4 GB,4.0,64 GB,64.0
2,6 GB,6.0,128 GB,128.0
3,6 GB,6.0,128 GB,128.0
4,4 GB,4.0,64 GB,64.0


In [85]:
# Remove the old columns such as Memory & Storage
df.drop(columns=['Storage','Memory'],inplace=True)

##### **4.6 Rating**

In [86]:
print("Unique values:",df['Rating'].unique(),'\n')
print("Data type:", df['Rating'].dtype, '\n')
print("Null values:",df['Rating'].isnull().sum())

Unique values: [4.5 4.3 4.4 4.2 nan 4.  4.6 3.8 3.  4.1 3.7 3.1 4.7 3.9 3.4 3.3 3.6 3.5
 3.2 2.7 2.8 5.  2.4 2.3 4.9 4.8] 

Data type: float64 

Null values: 144


Insights:
* Rating column has '144' null values.
* Droping those will lead to data loss because we have only 3114 entries.
* Also it means that no customer has rated it, so it will be considered as '0'.


In [87]:
df['Rating'] = df['Rating'].fillna(0)

##### **4.7 Selling Price**

In [88]:
print("Unique values:",df['Selling Price'].nunique(),'\n')
print("Data type:", df['Selling Price'].dtype, '\n')
print("Null values:",df['Selling Price'].isnull().sum())

Unique values: 844 

Data type: int64 

Null values: 0


Insights:
* This column has no null values.

##### **4.8 Original Price**

In [89]:
print("Unique values:",df['Original Price'].nunique(),'\n')
print("Data type:", df['Original Price'].dtype, '\n')
print("Null values:",df['Original Price'].isnull().sum())

Unique values: 819 

Data type: int64 

Null values: 0


Insights:

* This column has no null values.

#### **5. Checking after cleaning**

In [93]:
df.head(10)

,Brand,Model,Color,Rating,Selling Price,Original Price,Memory_GB,Storage_GB
0,OPPO,A53,Moonlight Black,4.5,11990,15990,4.0,64.0
1,OPPO,A53,Mint Cream,4.5,11990,15990,4.0,64.0
2,OPPO,A53,Moonlight Black,4.3,13990,17990,6.0,128.0
3,OPPO,A53,Mint Cream,4.3,13990,17990,6.0,128.0
4,OPPO,A53,Electric Black,4.5,11990,15990,4.0,64.0
5,OPPO,A53,Electric Black,4.3,13990,17990,6.0,128.0
6,OPPO,A12,Deep Blue,4.4,10490,11990,4.0,64.0
7,OPPO,A12,Black,4.4,9490,10990,3.0,32.0
8,OPPO,A12,Blue,4.4,9490,10990,3.0,32.0
9,OPPO,A12,Flowing Silver,4.4,9490,10990,3.0,32.0


In [94]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3114 entries, 0 to 3113
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Brand           3114 non-null   object 
 1   Model           3114 non-null   object 
 2   Color           3114 non-null   object 
 3   Rating          3114 non-null   float64
 4   Selling Price   3114 non-null   int64  
 5   Original Price  3114 non-null   int64  
 6   Memory_GB       3114 non-null   float64
 7   Storage_GB      3114 non-null   float64
dtypes: float64(3), int64(2), object(3)
memory usage: 194.8+ KB


#### **6. Feature Engineering**

Following are the new columns needed for our further analysis:

* 1. discount_percentage
* 2. price_gap
* 3. value_score
* 4. price_per_GB_RAM
* 5. price_per_GB_Storage
* 6. premium_flag
* 7. budget_flag

##### **6.1 Discount percentage**

`discount% = ((Original Price - Selling Price) / Original Price) * 100`


In [101]:
df['discount_percentage'] = (((df['Original Price'] - df['Selling Price'])/df['Original Price'])*100).round(2)

In [102]:
df['discount_percentage'].head(2)

,discount_percentage
0,25.02
1,25.02


##### **6.2 Price gap**

`price_gap = Original Price - Selling Price`


In [103]:
df['price_gap'] = df['Original Price'] - df['Selling Price']

In [104]:
df['price_gap'].head(2)

,price_gap
0,4000
1,4000


##### **6.3 Price per GB RAM**

`price_per_GB_RAM = Selling Price / Memory_GB`

* If Memory_GB = 0 (unknown), result becomes NaN
We will replace such cases with 0.


In [106]:
df['price_per_GB_RAM'] = (df['Selling Price'] / df['Memory_GB']).fillna(0).round(2)

In [107]:
df['price_per_GB_RAM'].head(2)

,price_per_GB_RAM
0,2997.5
1,2997.5


##### **6.4 Price per GB Storage**

`price_per_GB_storage = Selling Price / Storage_GB`

* If Storage_GB = 0 (unknown), result becomes NaN
We will replace such cases with 0.


In [108]:
df['price_per_GB_storage'] = (df['Selling Price'] / df['Storage_GB']).fillna(0).round(2)

In [109]:
df['price_per_GB_storage'].head(2)

,price_per_GB_storage
0,187.34
1,187.34


##### **6.5 Premium Flag**

Flag:

1 = premium phone (₹30,000+)

0 = not premium

In [112]:
df['Selling Price'].describe()

,Selling Price
count,3114.000000
mean,26436.625562
std,30066.892622
min,1000.000000
25%,9990.000000
50%,15000.000000
75%,28999.000000
max,179900.000000


**Why premium = +₹30,000?**

* The 75th percentile price in this dataset is ₹28,999, meaning phones above ₹30,000 fall into the top 25% most expensive models.
* This matches the Indian market standard where smartphones ₹30,000+ are considered premium.

In [113]:
df['premium_flag'] = (df['Selling Price'] >= 30000).astype(int)

In [114]:
df['premium_flag'].head(2)

,premium_flag
0,0
1,0


##### **6.6 Budget flag**

Flag:

1 = budget phone (less than ₹10,000)

0 = otherwise

**Why budget = below ₹10,000?**

* The 25th percentile price in this dataset is ₹9,990, meaning phones under ₹10,000 fall into the lowest-priced segment.
* This matches the Indian market standard where smartphones below ₹10,000 are classified as budget phones.

In [115]:
df['budget_flag'] = (df['Selling Price'] < 10000).astype(int)

In [116]:
df['budget_flag'].head(2)

,budget_flag
0,0
1,0


In [117]:
df.head(5)

,Brand,Model,Color,Rating,Selling Price,Original Price,Memory_GB,Storage_GB,discount_percentage,price_gap,price_per_GB_RAM,price_per_GB_storage,premium_flag,budget_flag
0,OPPO,A53,Moonlight Black,4.5,11990,15990,4.0,64.0,25.02,4000,2997.50,187.34,0,0
1,OPPO,A53,Mint Cream,4.5,11990,15990,4.0,64.0,25.02,4000,2997.50,187.34,0,0
2,OPPO,A53,Moonlight Black,4.3,13990,17990,6.0,128.0,22.23,4000,2331.67,109.30,0,0
3,OPPO,A53,Mint Cream,4.3,13990,17990,6.0,128.0,22.23,4000,2331.67,109.30,0,0
4,OPPO,A53,Electric Black,4.5,11990,15990,4.0,64.0,25.02,4000,2997.50,187.34,0,0


#### **7. Saving the Clean dataset**

In [120]:
clean_path = r'C:\Users\nagas\flipkart_price_tracker\data\cleaned\flipkart_mobiles_cleaned.csv'

df.to_csv(clean_path, index=False)

clean_path

'C:\\Users\\nagas\\flipkart_price_tracker\\data\\cleaned\\flipkart_mobiles_cleaned.csv'

#### **8.Viewing the cleaned schema**

In [121]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3114 entries, 0 to 3113
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Brand                 3114 non-null   object 
 1   Model                 3114 non-null   object 
 2   Color                 3114 non-null   object 
 3   Rating                3114 non-null   float64
 4   Selling Price         3114 non-null   int64  
 5   Original Price        3114 non-null   int64  
 6   Memory_GB             3114 non-null   float64
 7   Storage_GB            3114 non-null   float64
 8   discount_percentage   3114 non-null   float64
 9   price_gap             3114 non-null   int64  
 10  price_per_GB_RAM      3114 non-null   float64
 11  price_per_GB_storage  3114 non-null   float64
 12  premium_flag          3114 non-null   int64  
 13  budget_flag           3114 non-null   int64  
dtypes: float64(6), int64(5), object(3)
memory usage: 340.7+ KB
